# Simple pipeline with MLflow model tracking

This notebook builds an Iris classification pipeline and uses MLflow to track its runs and models. The `add_env_vars_to_tasks` function provides MLflow credentials to the pipeline components.

## MLflow Setup

Pipeline components run in separate pods, so they read MLflow credentials from
the `mlflow-credentials` Kubernetes secret. First, [create a Personal Access Token
in MLflow](/mlflow/oidc/ui/auth) and copy it. Then run the credential setup cell
below. The script prompts for your MLflow tracking URI, email address, and token,
then creates or updates the `mlflow-credentials` Kubernetes Secret for you.

If the Secret already exists, the script asks before replacing it and leaves it
unchanged by default. The separate `%run -n` cell only loads the helper functions
used later; it does not prompt or modify the Secret. Automated tests skip the
interactive setup cell and use the preconfigured Secret validated during preflight.

## How it works

The `add_env_vars_to_tasks` function injects the credentials from the Kubernetes secret into each component that uses MLflow.

The pipeline preprocesses data, trains and logs a model, and makes predictions on test data. It passes MLflow run information between components so later steps can continue the same run.

In [ ]:
# Prompts for your MLflow URI, email, and PAT, then creates or updates the Secret.
setup_mlflow_credentials()

In [ ]:
# Following dependency is available by default in prokube notebooks:
# !pip install -q kfp[all]==2.15.0

In [ ]:
# Load the MLflow credential helpers without starting interactive setup.
%run -n ~/examples/src/pk_helpers/mlflow_credentials.py

In [ ]:
import os

import kfp.dsl as dsl
from kfp.client import Client
from kfp.dsl import Dataset, Input, Model, Output, Artifact
from kfp.kubernetes import use_secret_as_env

In [ ]:
# Check the credentials before submitting the pipeline so a missing secret
# produces a clear error here rather than inside a pipeline component.
require_mlflow_secret()

In [ ]:
def add_env_vars_to_tasks(task_list: list[dsl.PipelineTask]) -> None:
    """Add MLflow credentials to the tasks that use MLflow."""
    for task in task_list:
        use_secret_as_env(
            task,
            secret_name="mlflow-credentials",
            secret_key_to_env={
                "MLFLOW_TRACKING_URI": "MLFLOW_TRACKING_URI",
                "MLFLOW_TRACKING_USERNAME": "MLFLOW_TRACKING_USERNAME",
                "MLFLOW_TRACKING_PASSWORD": "MLFLOW_TRACKING_PASSWORD",
            }
        )


    # if your installation's TLS certificate doesn't contain the full trust chain, you should uncomment the below line
    # task.set_env_variable("MLFLOW_TRACKING_INSECURE_TLS", "true")

## Preprocess data

In [ ]:
@dsl.component(
    packages_to_install=["pandas", "pyarrow", "scikit-learn"],
    base_image="python:3.11",
)
def preprocess_data(
    x_train_df: Output[Dataset],
    y_train_df: Output[Dataset],
    x_test_df: Output[Dataset],
    y_test_df: Output[Dataset],
    test_size: float = 0.2,
    seed: int = 42,
):
    """Reads iris data and writes it to pipeline artifacts as parquet."""
    from sklearn import datasets
    from sklearn.model_selection import train_test_split

    df = datasets.load_iris(as_frame=True)
    x = df.data
    y = df.target.to_frame()

    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=test_size, random_state=seed
    )

    for obj, artifact in zip(
        (x_train, x_test, y_train, y_test),
        (x_train_df, x_test_df, y_train_df, y_test_df)
    ):
        obj.to_parquet(artifact.path)

## Train and log model

As an example of how to use MLflow with pipelines, this notebook saves MLflow run parameters as a dict. This dict can be loaded from other KFP tasks.

In [ ]:


@dsl.component(
    packages_to_install=["pandas", "pyarrow", "scikit-learn", "mlflow==3.10.0", "boto3"],
    base_image="python:3.11",
)
def train_and_log_model(
    x_train: Input[Dataset],
    y_train: Input[Dataset],
    seed: int = 42,
) -> dict:
    
    import os

    import mlflow
    import pandas as pd
    from mlflow.models import infer_signature
    from sklearn.linear_model import LogisticRegression

    username = os.getenv('MLFLOW_TRACKING_USERNAME').split('@')[0]


    x_train = pd.read_parquet(x_train.path)
    y_train = pd.read_parquet(y_train.path)

    # Define the model hyperparameters
    params = {
        "solver": "lbfgs",
        "max_iter": 1000,
        "random_state": seed,
    }

    # Train the model
    lr = LogisticRegression(**params)
    lr.fit(x_train, y_train)

    # Create MLflow Experiment name
    mlflow.set_experiment(f"MLflow Quickstart with KFP {username}")

    # Start an MLflow run
    with mlflow.start_run() as run:
        # Log the hyperparameters
        mlflow.log_params(params)

        # Set a tag that we can use to remind ourselves what this run was for
        mlflow.set_tag("Training Info", "Basic LR model for iris data, KFP")

        # Infer the model signature
        signature = infer_signature(x_train, lr.predict(x_train))



        # Log the model
        model_info = mlflow.sklearn.log_model(
            sk_model=lr,
            name="iris-model",
            signature=signature,
            input_example=x_train,
            registered_model_name=f"tracking-quickstart-pipeline-{username}"
        )
    
    # Save run as dict
    return run.to_dictionary()



## Load the model from MLflow and make predictions

This component loads model saved to MLflow based on the run ID. Requires the dictionary with MLflow run information as an input.

In [ ]:
@dsl.component(
    packages_to_install=["pandas", "pyarrow", "scikit-learn", "mlflow==3.10.0", "boto3"],
    base_image="python:3.11",
)
def predict(
    x_test: Input[Dataset],
    y_test: Input[Dataset],
    mlflow_run: dict,
):
    import os

    import mlflow
    import pandas as pd
    from sklearn.metrics import accuracy_score


    # Load trained model
    run_id = mlflow_run["info"]["run_id"]
    model_path = f"runs:/{run_id}/iris-model"  # model name (iris-model) corresponds to artifact path 
    model = mlflow.sklearn.load_model(model_path)

    # Load test data
    x_test = pd.read_parquet(x_test.path)
    y_test = pd.read_parquet(y_test.path)

    # Predict on the test set
    y_pred = model.predict(x_test)

    # Calculate metric
    accuracy = accuracy_score(y_test, y_pred)
    
    with mlflow.start_run(run_id=run_id):
        # Log the loss metric
        mlflow.log_metric("accuracy", accuracy)

## Build and run pipeline

In [ ]:
@dsl.pipeline
def simple_pipeline():

    # Step 1: Preprocess the data
    preprocess_data_task = preprocess_data()

    # Step 2: Train the model and add necessary env vars
    train_and_log_model_task = train_and_log_model(
        x_train=preprocess_data_task.outputs['x_train_df'],
        y_train=preprocess_data_task.outputs['y_train_df'],
    )

    # Step 3: Predict on test data
    predict_task = predict(
        x_test=preprocess_data_task.outputs['x_test_df'],
        y_test=preprocess_data_task.outputs['y_test_df'],
        mlflow_run=train_and_log_model_task.output,
    )
    
    # Add env vars
    add_env_vars_to_tasks([train_and_log_model_task, predict_task])


# Initialize the Kubeflow Pipelines client
client = Client()

# Create a new run from the pipeline function
client.create_run_from_pipeline_func(
    simple_pipeline,
    experiment_name="iris-dataset-classification",
    enable_caching=True,
)

# kfp.compiler.Compiler().compile(simple_pipeline, 'simple_pipeline.yaml')